# Data Mining Assignment 3 — Human Activity Recognition Pipeline

This notebook builds a complete machine learning pipeline for the Kaggle HAR competition.

The goal is to predict one activity label between **0 and 5** for each 5-minute accelerometer sequence.  
Each CSV file represents one sequence of 300 seconds. Each row contains aggregated accelerometer statistics for one second.

The notebook includes:

1. Data loading from the expected `data/` folder structure.
2. Exploratory checks.
3. Feature engineering.
4. Baseline 1: statistical time-series features.
5. Baseline 2: enriched temporal + frequency-domain features.
6. Cross-validation comparison using macro F1-score.
7. Final training and Kaggle submission generation.

Expected project structure:

```text
your_notebook.ipynb
data/
├── train/
│   └── ...
├── test/
│   └── test/
│       └── User_061/
│           ├── 11021.csv
│           ├── 11022.csv
│           └── ...
└── sample_submission.csv
```


In [1]:
# ============================================================
# 1. Imports and global configuration
# ============================================================

from pathlib import Path
import warnings
import math
import os

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from scipy.stats import skew, kurtosis
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# The notebook is expected to be located at the project root.
ROOT_DIR = Path(".")
DATA_DIR = ROOT_DIR / "data"

TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

OUTPUT_DIR = ROOT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("DATA_DIR:", DATA_DIR.resolve())
print("TRAIN_DIR exists:", TRAIN_DIR.exists())
print("TEST_DIR exists:", TEST_DIR.exists())
print("Sample submission exists:", SAMPLE_SUBMISSION_PATH.exists())


DATA_DIR: C:\Users\titou\Documents\ISEN\M1\Taiwan\Cours\Data Mining\Assignment 3\data
TRAIN_DIR exists: True
TEST_DIR exists: True
Sample submission exists: True


## 2. Data loading

The dataset is organized as many small CSV files.  
Each CSV file corresponds to one 5-minute window and therefore to one final activity label.

For the training set, the label is expected to be available inside each CSV file, usually repeated on each row.  
For the test set, labels are unknown and must be predicted.

The loader below searches recursively, so it supports structures such as:

```text
data/test/test/User_061/11021.csv
data/train/train/User_001/00001.csv
```

or similar nested folders.


In [2]:
# ============================================================
# 2. Recursive CSV discovery and loading utilities
# ============================================================

def find_csv_files(base_dir: Path):
    """Return all CSV files recursively under a directory."""
    if not base_dir.exists():
        return []
    return sorted([p for p in base_dir.rglob("*.csv") if p.is_file()])


def read_sequence_csv(path: Path) -> pd.DataFrame:
    """Read one accelerometer sequence CSV."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    return df


def infer_file_id(df: pd.DataFrame, path: Path) -> int:
    """Infer file_id from the CSV content if available, otherwise from the filename."""
    if "file_id" in df.columns:
        unique_ids = df["file_id"].dropna().unique()
        if len(unique_ids) > 0:
            return int(unique_ids[0])
    return int(path.stem)


def infer_user_id(path: Path) -> str:
    """Infer user ID from the parent folder name when possible."""
    for part in reversed(path.parts):
        if part.lower().startswith("user_"):
            return part
    return path.parent.name


def load_sequences(base_dir: Path, has_labels: bool):
    """
    Load all sequence CSV files from a base directory.

    Returns a list of dictionaries:
    {
        "path": Path,
        "file_id": int,
        "user_id": str,
        "label": int or None,
        "df": DataFrame
    }
    """
    csv_files = find_csv_files(base_dir)
    sequences = []

    for path in csv_files:
        df = read_sequence_csv(path)
        file_id = infer_file_id(df, path)
        user_id = infer_user_id(path)

        label = None
        if has_labels:
            if "label" not in df.columns:
                raise ValueError(f"Training file has no 'label' column: {path}")
            label_values = df["label"].dropna().astype(int)
            if len(label_values) == 0:
                raise ValueError(f"Training file has empty labels: {path}")
            label = int(label_values.mode().iloc[0])

        sequences.append({
            "path": path,
            "file_id": file_id,
            "user_id": user_id,
            "label": label,
            "df": df
        })

    return sequences


train_sequences = load_sequences(TRAIN_DIR, has_labels=True)
test_sequences = load_sequences(TEST_DIR, has_labels=False)

print(f"Number of training sequences: {len(train_sequences)}")
print(f"Number of test sequences: {len(test_sequences)}")

if len(train_sequences) > 0:
    print("\nExample training file:")
    print(train_sequences[0]["path"])
    display(train_sequences[0]["df"].head())

if len(test_sequences) > 0:
    print("\nExample test file:")
    print(test_sequences[0]["path"])
    display(test_sequences[0]["df"].head())


Number of training sequences: 11020
Number of test sequences: 6849

Example training file:
data\train\train\User_001\00001.csv


,index,mean_x,mean_y,mean_z,std_x,std_y,std_z,label,file_id
0,0,-0.467629,-0.537854,0.657240,3.733827e-03,0.007097,0.004734,0,1
1,1,-0.466690,-0.547035,0.654931,1.115816e-16,0.005082,0.006511,0,1
2,2,-0.467316,-0.535987,0.658472,3.080919e-03,0.005875,0.002188,0,1
3,3,-0.468880,-0.533341,0.658472,5.455416e-03,0.000000,0.000000,0,1
4,4,-0.470288,-0.533341,0.658472,6.616433e-03,0.000000,0.000000,0,1



Example test file:
data\test\test\User_061\11021.csv


,index,mean_x,mean_y,mean_z,std_x,std_y,std_z,file_id
0,0,-0.373418,0.460280,0.791259,0.010760,0.002213,0.006573,11021
1,1,-0.370264,0.459337,0.796401,0.012162,0.003098,0.008336,11021
2,2,-0.371999,0.458550,0.795919,0.011466,0.005047,0.007944,11021
3,3,-0.368845,0.462954,0.792062,0.008615,0.006202,0.006483,11021
4,4,-0.369791,0.459494,0.796241,0.011496,0.003503,0.007996,11021


## 3. Preliminary analysis

Before building models, we check basic properties of the dataset:

- number of files;
- number of rows per sequence;
- label distribution;
- available columns;
- whether the sequence length is always 300 seconds.

This part is useful for the report because it justifies the method design.


In [3]:
# ============================================================
# 3. Preliminary dataset checks
# ============================================================

def summarize_sequences(sequences, name):
    records = []
    for item in sequences:
        df = item["df"]
        records.append({
            "file_id": item["file_id"],
            "user_id": item["user_id"],
            "path": str(item["path"]),
            "n_rows": len(df),
            "label": item["label"]
        })
    summary = pd.DataFrame(records)
    print(f"{name} summary shape:", summary.shape)
    display(summary.head())
    return summary


train_summary = summarize_sequences(train_sequences, "Train")
test_summary = summarize_sequences(test_sequences, "Test")

if len(train_summary) > 0:
    print("\nLabel distribution:")
    display(train_summary["label"].value_counts().sort_index().to_frame("count"))

    print("\nSequence length distribution:")
    display(train_summary["n_rows"].value_counts().sort_index().to_frame("count"))

if len(test_summary) > 0:
    print("\nTest sequence length distribution:")
    display(test_summary["n_rows"].value_counts().sort_index().to_frame("count"))

if len(train_sequences) > 0:
    print("\nColumns in first training file:")
    print(list(train_sequences[0]["df"].columns))


Train summary shape: (11020, 5)


,file_id,user_id,path,n_rows,label
0,1,User_001,data\train\train\User_001\00001.csv,300,0
1,2,User_001,data\train\train\User_001\00002.csv,300,0
2,3,User_001,data\train\train\User_001\00003.csv,300,0
3,4,User_001,data\train\train\User_001\00004.csv,300,0
4,5,User_001,data\train\train\User_001\00005.csv,300,0


Test summary shape: (6849, 5)


,file_id,user_id,path,n_rows,label
0,11021,User_061,data\test\test\User_061\11021.csv,300,None
1,11022,User_061,data\test\test\User_061\11022.csv,300,None
2,11023,User_061,data\test\test\User_061\11023.csv,300,None
3,11024,User_061,data\test\test\User_061\11024.csv,300,None
4,11025,User_061,data\test\test\User_061\11025.csv,300,None



Label distribution:


...


Sequence length distribution:


,count
n_rows,
300,11020



Test sequence length distribution:


,count
n_rows,
300,6849



Columns in first training file:
['index', 'mean_x', 'mean_y', 'mean_z', 'std_x', 'std_y', 'std_z', 'label', 'file_id']


## 4. Feature engineering strategy

Each original CSV contains 300 rows, one row per second.  
Most classical machine learning models cannot directly use a sequence-shaped input, so we convert each file into one fixed-size feature vector.

This notebook evaluates three feature-engineering baselines:

- **Baseline 1 — Statistical features**: global mean, standard deviation, quantiles, min/max, skewness, kurtosis and first-difference statistics.
- **Baseline 2 — Temporal + FFT features**: Baseline 1 plus temporal chunks, trends, autocorrelation, rolling statistics and frequency-domain descriptors.
- **Baseline 3 — Improved HAR features**: a stronger handcrafted feature set focused on acceleration magnitude, jerk magnitude, dominant frequencies, spectral entropy and finer temporal chunks.

The goal is not only to improve the Kaggle score, but also to create a clean ablation study for the report.


In [4]:
# ============================================================
# 4. Feature engineering functions
# ============================================================

SENSOR_COLS = ["mean_x", "mean_y", "mean_z", "std_x", "std_y", "std_z"]
MEAN_AXIS_COLS = ["mean_x", "mean_y", "mean_z"]


def safe_array(series):
    """Convert a pandas Series to a clean float numpy array."""
    return pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy(dtype=float)


def add_basic_stats(features, values, prefix):
    """Add robust statistical descriptors for one numeric signal."""
    values = np.asarray(values, dtype=float)

    features[f"{prefix}_mean"] = np.mean(values)
    features[f"{prefix}_std"] = np.std(values)
    features[f"{prefix}_min"] = np.min(values)
    features[f"{prefix}_max"] = np.max(values)
    features[f"{prefix}_median"] = np.median(values)
    features[f"{prefix}_q25"] = np.percentile(values, 25)
    features[f"{prefix}_q75"] = np.percentile(values, 75)
    features[f"{prefix}_iqr"] = features[f"{prefix}_q75"] - features[f"{prefix}_q25"]
    features[f"{prefix}_range"] = features[f"{prefix}_max"] - features[f"{prefix}_min"]

    if np.std(values) > 1e-12:
        features[f"{prefix}_skew"] = skew(values)
        features[f"{prefix}_kurtosis"] = kurtosis(values)
    else:
        features[f"{prefix}_skew"] = 0.0
        features[f"{prefix}_kurtosis"] = 0.0

    return features


def add_diff_stats(features, values, prefix):
    """Add statistics on first differences to capture local motion changes."""
    values = np.asarray(values, dtype=float)
    if len(values) < 2:
        diffs = np.array([0.0])
    else:
        diffs = np.diff(values)

    features[f"{prefix}_diff_mean_abs"] = np.mean(np.abs(diffs))
    features[f"{prefix}_diff_std"] = np.std(diffs)
    features[f"{prefix}_diff_max_abs"] = np.max(np.abs(diffs))
    features[f"{prefix}_diff_energy"] = np.mean(diffs ** 2)

    return features


def extract_baseline1_features(df: pd.DataFrame) -> dict:
    """
    Baseline 1:
    Global statistical features from each sensor column.
    """
    features = {}

    available_cols = [c for c in SENSOR_COLS if c in df.columns]

    for col in available_cols:
        values = safe_array(df[col])
        add_basic_stats(features, values, col)
        add_diff_stats(features, values, col)

    # Signal magnitude from mean acceleration axes.
    if all(c in df.columns for c in MEAN_AXIS_COLS):
        x = safe_array(df["mean_x"])
        y = safe_array(df["mean_y"])
        z = safe_array(df["mean_z"])

        magnitude = np.sqrt(x ** 2 + y ** 2 + z ** 2)
        add_basic_stats(features, magnitude, "acc_magnitude")
        add_diff_stats(features, magnitude, "acc_magnitude")

        # Energy-like features.
        features["acc_energy_mean"] = np.mean(x ** 2 + y ** 2 + z ** 2)
        features["acc_energy_std"] = np.std(x ** 2 + y ** 2 + z ** 2)

        # Correlations between axes.
        axis_values = {"x": x, "y": y, "z": z}
        for a_name, a_values in axis_values.items():
            for b_name, b_values in axis_values.items():
                if a_name < b_name:
                    if np.std(a_values) > 1e-12 and np.std(b_values) > 1e-12:
                        corr = np.corrcoef(a_values, b_values)[0, 1]
                    else:
                        corr = 0.0
                    features[f"corr_{a_name}_{b_name}"] = corr

    return features


def autocorr(values, lag):
    """Compute autocorrelation at a given lag."""
    values = np.asarray(values, dtype=float)
    if len(values) <= lag:
        return 0.0

    a = values[:-lag]
    b = values[lag:]

    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return 0.0

    return float(np.corrcoef(a, b)[0, 1])


def spectral_entropy(power):
    """Compute normalized spectral entropy from FFT power."""
    power = np.asarray(power, dtype=float)
    total = np.sum(power)

    if total <= 1e-12:
        return 0.0

    p = power / total
    p = p[p > 0]
    entropy = -np.sum(p * np.log(p))
    return float(entropy / np.log(len(power))) if len(power) > 1 else 0.0


def add_fft_features(features, values, prefix):
    """Add frequency-domain features using FFT power spectrum."""
    values = np.asarray(values, dtype=float)
    values = values - np.mean(values)

    if len(values) < 4 or np.std(values) < 1e-12:
        features[f"{prefix}_fft_total_power"] = 0.0
        features[f"{prefix}_fft_dom_freq_idx"] = 0
        features[f"{prefix}_fft_dom_power_ratio"] = 0.0
        features[f"{prefix}_fft_low_power_ratio"] = 0.0
        features[f"{prefix}_fft_mid_power_ratio"] = 0.0
        features[f"{prefix}_fft_high_power_ratio"] = 0.0
        features[f"{prefix}_fft_spectral_entropy"] = 0.0
        return features

    fft_values = np.fft.rfft(values)
    power = np.abs(fft_values) ** 2

    # Remove DC component for dominant frequency analysis.
    power_no_dc = power.copy()
    if len(power_no_dc) > 0:
        power_no_dc[0] = 0.0

    total_power = np.sum(power_no_dc)
    dom_idx = int(np.argmax(power_no_dc))
    dom_power = power_no_dc[dom_idx]

    n = len(power_no_dc)
    low = power_no_dc[: max(1, n // 6)]
    mid = power_no_dc[max(1, n // 6): max(2, n // 3)]
    high = power_no_dc[max(2, n // 3):]

    features[f"{prefix}_fft_total_power"] = float(total_power)
    features[f"{prefix}_fft_dom_freq_idx"] = dom_idx
    features[f"{prefix}_fft_dom_power_ratio"] = float(dom_power / total_power) if total_power > 1e-12 else 0.0
    features[f"{prefix}_fft_low_power_ratio"] = float(np.sum(low) / total_power) if total_power > 1e-12 else 0.0
    features[f"{prefix}_fft_mid_power_ratio"] = float(np.sum(mid) / total_power) if total_power > 1e-12 else 0.0
    features[f"{prefix}_fft_high_power_ratio"] = float(np.sum(high) / total_power) if total_power > 1e-12 else 0.0
    features[f"{prefix}_fft_spectral_entropy"] = spectral_entropy(power_no_dc)

    return features


def extract_baseline2_features(df: pd.DataFrame) -> dict:
    """
    Baseline 2:
    Baseline 1 + temporal chunks + trends + autocorrelation + FFT features.
    """
    features = extract_baseline1_features(df)

    available_cols = [c for c in SENSOR_COLS if c in df.columns]

    for col in available_cols:
        values = safe_array(df[col])
        n = len(values)

        # Temporal chunks: split the 5-minute sequence into four parts.
        chunks = np.array_split(values, 4)
        for i, chunk in enumerate(chunks):
            features[f"{col}_chunk{i+1}_mean"] = np.mean(chunk)
            features[f"{col}_chunk{i+1}_std"] = np.std(chunk)
            features[f"{col}_chunk{i+1}_min"] = np.min(chunk)
            features[f"{col}_chunk{i+1}_max"] = np.max(chunk)

        # Beginning vs ending comparison.
        features[f"{col}_last_minus_first_mean"] = np.mean(chunks[-1]) - np.mean(chunks[0])
        features[f"{col}_last_minus_first_std"] = np.std(chunks[-1]) - np.std(chunks[0])

        # Linear trend over time.
        if n >= 2:
            t = np.arange(n)
            slope = np.polyfit(t, values, 1)[0]
            features[f"{col}_linear_trend"] = float(slope)
        else:
            features[f"{col}_linear_trend"] = 0.0

        # Autocorrelation features.
        for lag in [1, 2, 5, 10, 20]:
            features[f"{col}_autocorr_lag{lag}"] = autocorr(values, lag)

        # Rolling-window features.
        rolling_window = min(10, max(2, n // 10))
        rolling_mean = pd.Series(values).rolling(rolling_window, min_periods=1).mean().to_numpy()
        rolling_std = pd.Series(values).rolling(rolling_window, min_periods=1).std().fillna(0).to_numpy()

        features[f"{col}_rolling_mean_std"] = np.std(rolling_mean)
        features[f"{col}_rolling_std_mean"] = np.mean(rolling_std)
        features[f"{col}_rolling_std_max"] = np.max(rolling_std)

        # Frequency-domain features.
        add_fft_features(features, values, col)

    # Additional magnitude frequency features.
    if all(c in df.columns for c in MEAN_AXIS_COLS):
        x = safe_array(df["mean_x"])
        y = safe_array(df["mean_y"])
        z = safe_array(df["mean_z"])
        magnitude = np.sqrt(x ** 2 + y ** 2 + z ** 2)
        add_fft_features(features, magnitude, "acc_magnitude")

    return features




def safe_corr(a, b):
    """Compute a correlation value safely, even for constant signals."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return 0.0

    return float(np.corrcoef(a, b)[0, 1])


def add_energy_features(features, values, prefix):
    """Add simple energy and amplitude features."""
    values = np.asarray(values, dtype=float)

    features[f"{prefix}_energy_mean"] = float(np.mean(values ** 2))
    features[f"{prefix}_abs_mean"] = float(np.mean(np.abs(values)))
    features[f"{prefix}_rms"] = float(np.sqrt(np.mean(values ** 2)))
    features[f"{prefix}_zero_crossings"] = int(np.sum(np.diff(np.signbit(values - np.mean(values)))))

    return features


def add_dominant_frequency_features(features, values, prefix):
    """
    Add frequency-domain features using real frequencies.

    The dataset is aggregated at 1 Hz, so the sampling period is 1 second.
    These features are useful for HAR because walking, running or repetitive
    movements often differ by their dominant rhythm.
    """
    values = np.asarray(values, dtype=float)
    values = values - np.mean(values)

    if len(values) < 4 or np.std(values) < 1e-12:
        features[f"{prefix}_dom_freq_hz"] = 0.0
        features[f"{prefix}_dom_power"] = 0.0
        features[f"{prefix}_spectral_centroid"] = 0.0
        features[f"{prefix}_spectral_entropy_real"] = 0.0
        return features

    fft_values = np.fft.rfft(values)
    power = np.abs(fft_values) ** 2
    freqs = np.fft.rfftfreq(len(values), d=1.0)

    # Remove DC component because it mainly represents the average level.
    power[0] = 0.0
    total_power = np.sum(power)

    if total_power <= 1e-12:
        features[f"{prefix}_dom_freq_hz"] = 0.0
        features[f"{prefix}_dom_power"] = 0.0
        features[f"{prefix}_spectral_centroid"] = 0.0
        features[f"{prefix}_spectral_entropy_real"] = 0.0
        return features

    dom_idx = int(np.argmax(power))
    prob = power / total_power
    positive_prob = prob[prob > 0]

    features[f"{prefix}_dom_freq_hz"] = float(freqs[dom_idx])
    features[f"{prefix}_dom_power"] = float(power[dom_idx])
    features[f"{prefix}_spectral_centroid"] = float(np.sum(freqs * power) / total_power)
    features[f"{prefix}_spectral_entropy_real"] = float(-np.sum(positive_prob * np.log(positive_prob)) / np.log(len(power)))

    return features


def extract_improved_features(df: pd.DataFrame) -> dict:
    """
    Baseline 3:
    Improved HAR features.

    This extractor keeps the useful features from Baseline 2 and adds:
    - acceleration magnitude descriptors;
    - jerk descriptors, representing rapid motion changes;
    - stronger frequency-domain features with dominant frequency in Hz;
    - spectral entropy and spectral centroid;
    - finer temporal chunks to preserve more of the 5-minute dynamics.
    """
    features = extract_baseline2_features(df)

    if all(c in df.columns for c in MEAN_AXIS_COLS):
        x = safe_array(df["mean_x"])
        y = safe_array(df["mean_y"])
        z = safe_array(df["mean_z"])

        magnitude = np.sqrt(x ** 2 + y ** 2 + z ** 2)
        add_basic_stats(features, magnitude, "mag")
        add_diff_stats(features, magnitude, "mag")
        add_energy_features(features, magnitude, "mag")
        add_dominant_frequency_features(features, magnitude, "mag")

        # Jerk captures movement changes between consecutive seconds.
        jerk_x = np.diff(x) if len(x) >= 2 else np.array([0.0])
        jerk_y = np.diff(y) if len(y) >= 2 else np.array([0.0])
        jerk_z = np.diff(z) if len(z) >= 2 else np.array([0.0])
        jerk_mag = np.sqrt(jerk_x ** 2 + jerk_y ** 2 + jerk_z ** 2)

        for name, values in {
            "jerk_x": jerk_x,
            "jerk_y": jerk_y,
            "jerk_z": jerk_z,
            "jerk_mag": jerk_mag,
        }.items():
            add_basic_stats(features, values, name)
            add_energy_features(features, values, name)
            add_dominant_frequency_features(features, values, name)

        # Extra cross-axis relationships.
        features["mean_corr_xy"] = safe_corr(x, y)
        features["mean_corr_xz"] = safe_corr(x, z)
        features["mean_corr_yz"] = safe_corr(y, z)
        features["jerk_corr_xy"] = safe_corr(jerk_x, jerk_y)
        features["jerk_corr_xz"] = safe_corr(jerk_x, jerk_z)
        features["jerk_corr_yz"] = safe_corr(jerk_y, jerk_z)

        # Finer temporal chunks. This keeps more temporal structure than the 4 chunks in Baseline 2.
        n_chunks = 10
        chunk_indices = np.array_split(np.arange(len(df)), n_chunks)

        for i, idx in enumerate(chunk_indices, start=1):
            if len(idx) == 0:
                continue

            chunk_x = x[idx]
            chunk_y = y[idx]
            chunk_z = z[idx]
            chunk_mag = magnitude[idx]

            for prefix, values in {
                f"chunk{i}_mean_x": chunk_x,
                f"chunk{i}_mean_y": chunk_y,
                f"chunk{i}_mean_z": chunk_z,
                f"chunk{i}_mag": chunk_mag,
            }.items():
                features[f"{prefix}_mean"] = float(np.mean(values))
                features[f"{prefix}_std"] = float(np.std(values))
                features[f"{prefix}_min"] = float(np.min(values))
                features[f"{prefix}_max"] = float(np.max(values))
                features[f"{prefix}_energy"] = float(np.mean(values ** 2))

    # Dominant-frequency features for all original sensor columns.
    for col in [c for c in SENSOR_COLS if c in df.columns]:
        values = safe_array(df[col])
        add_energy_features(features, values, col)
        add_dominant_frequency_features(features, values, col)

    return features


def build_feature_table(sequences, feature_extractor):
    """Convert a list of sequences into a feature matrix with a progress bar."""
    rows = []

    for item in tqdm(
        sequences,
        total=len(sequences),
        desc=f"Building features ({feature_extractor.__name__})"
    ):
        feats = feature_extractor(item["df"])
        feats["file_id"] = item["file_id"]
        feats["user_id"] = item["user_id"]
        feats["label"] = item["label"]
        rows.append(feats)

    return pd.DataFrame(rows)


## 5. Build feature tables

We now create one row per CSV file.

- `train_b1`: Baseline 1 statistical feature table.
- `train_b2`: Baseline 2 temporal + FFT feature table.
- `train_b3`: Baseline 3 improved HAR feature table.
- `test_b1`, `test_b2`, `test_b3`: the same feature sets for the Kaggle test set.

The function uses `tqdm`, so you can monitor the progress while the thousands of CSV files are processed.


In [5]:
# ============================================================
# 5. Build baseline feature tables
# ============================================================

train_b1 = build_feature_table(train_sequences, extract_baseline1_features)
test_b1 = build_feature_table(test_sequences, extract_baseline1_features)

train_b2 = build_feature_table(train_sequences, extract_baseline2_features)
test_b2 = build_feature_table(test_sequences, extract_baseline2_features)

train_b3 = build_feature_table(train_sequences, extract_improved_features)
test_b3 = build_feature_table(test_sequences, extract_improved_features)

print("Baseline 1 train shape:", train_b1.shape)
print("Baseline 1 test shape:", test_b1.shape)
print("Baseline 2 train shape:", train_b2.shape)
print("Baseline 2 test shape:", test_b2.shape)
print("Baseline 3 train shape:", train_b3.shape)
print("Baseline 3 test shape:", test_b3.shape)

display(train_b1.head())
display(train_b2.head())
display(train_b3.head())


Building features (extract_baseline1_features):   0%|          | 0/11020 [00:00<?, ?it/s]

Building features (extract_baseline1_features):   0%|          | 0/6849 [00:00<?, ?it/s]

Building features (extract_baseline2_features):   0%|          | 0/11020 [00:00<?, ?it/s]

Building features (extract_baseline2_features):   0%|          | 0/6849 [00:00<?, ?it/s]

Building features (extract_improved_features):   0%|          | 0/11020 [00:00<?, ?it/s]

Building features (extract_improved_features):   0%|          | 0/6849 [00:00<?, ?it/s]

Baseline 1 train shape: (11020, 113)
Baseline 1 test shape: (6849, 113)
Baseline 2 train shape: (11020, 324)
Baseline 2 test shape: (6849, 324)
Baseline 3 train shape: (11020, 677)
Baseline 3 test shape: (6849, 677)


,mean_x_mean,mean_x_std,mean_x_min,mean_x_max,mean_x_median,mean_x_q25,mean_x_q75,mean_x_iqr,mean_x_range,mean_x_skew,...,acc_magnitude_diff_max_abs,acc_magnitude_diff_energy,acc_energy_mean,acc_energy_std,corr_x_y,corr_x_z,corr_y_z,file_id,user_id,label
0,-0.474944,0.003214,-0.480706,-0.466690,-0.475184,-0.477797,-0.472596,0.005201,0.014015,0.420654,...,0.004171,0.000002,0.943241,0.002114,-0.672841,-0.489820,0.898587,1,User_001,0
1,-0.477411,0.002954,-0.482270,-0.466157,-0.478000,-0.479541,-0.475456,0.004085,0.016113,0.698985,...,0.004646,0.000001,0.943198,0.001907,-0.540687,-0.070063,0.802157,2,User_001,0
2,-0.477836,0.002302,-0.481455,-0.472571,-0.478405,-0.479865,-0.475856,0.004009,0.008884,0.436866,...,0.003355,0.000001,0.943550,0.001524,-0.913854,-0.785442,0.881651,3,User_001,0
3,-0.476572,0.002982,-0.481614,-0.469691,-0.477044,-0.478612,-0.474071,0.004540,0.011923,0.259547,...,0.003531,0.000002,0.946386,0.002469,0.126408,0.548904,0.833940,4,User_001,0
4,-0.476293,0.002300,-0.480955,-0.471099,-0.476830,-0.478196,-0.474352,0.003844,0.009855,0.258560,...,0.002635,0.000001,0.948098,0.001559,-0.782942,-0.513768,0.602880,5,User_001,0


,mean_x_mean,mean_x_std,mean_x_min,mean_x_max,mean_x_median,mean_x_q25,mean_x_q75,mean_x_iqr,mean_x_range,mean_x_skew,...,acc_magnitude_fft_total_power,acc_magnitude_fft_dom_freq_idx,acc_magnitude_fft_dom_power_ratio,acc_magnitude_fft_low_power_ratio,acc_magnitude_fft_mid_power_ratio,acc_magnitude_fft_high_power_ratio,acc_magnitude_fft_spectral_entropy,file_id,user_id,label
0,-0.474944,0.003214,-0.480706,-0.466690,-0.475184,-0.477797,-0.472596,0.005201,0.014015,0.420654,...,0.053435,1,0.169438,0.277977,0.044920,0.677103,0.755914,1,User_001,0
1,-0.477411,0.002954,-0.482270,-0.466157,-0.478000,-0.479541,-0.475456,0.004085,0.016113,0.698985,...,0.043360,1,0.213118,0.365133,0.076894,0.557973,0.746142,2,User_001,0
2,-0.477836,0.002302,-0.481455,-0.472571,-0.478405,-0.479865,-0.475856,0.004009,0.008884,0.436866,...,0.027840,71,0.084175,0.114654,0.128460,0.756886,0.891579,3,User_001,0
3,-0.476572,0.002982,-0.481614,-0.469691,-0.477044,-0.478612,-0.474071,0.004540,0.011923,0.259547,...,0.072669,1,0.292350,0.461762,0.034716,0.503522,0.585807,4,User_001,0
4,-0.476293,0.002300,-0.480955,-0.471099,-0.476830,-0.478196,-0.474352,0.003844,0.009855,0.258560,...,0.029027,71,0.271566,0.084592,0.079735,0.835674,0.739153,5,User_001,0


,mean_x_mean,mean_x_std,mean_x_min,mean_x_max,mean_x_median,mean_x_q25,mean_x_q75,mean_x_iqr,mean_x_range,mean_x_skew,...,std_z_abs_mean,std_z_rms,std_z_zero_crossings,std_z_dom_freq_hz,std_z_dom_power,std_z_spectral_centroid,std_z_spectral_entropy_real,file_id,user_id,label
0,-0.474944,0.003214,-0.480706,-0.466690,-0.475184,-0.477797,-0.472596,0.005201,0.014015,0.420654,...,0.003545,0.004083,86,0.003333,0.053772,0.110548,0.645200,1,User_001,0
1,-0.477411,0.002954,-0.482270,-0.466157,-0.478000,-0.479541,-0.475456,0.004085,0.016113,0.698985,...,0.005853,0.006074,78,0.003333,0.050884,0.114661,0.539414,2,User_001,0
2,-0.477836,0.002302,-0.481455,-0.472571,-0.478405,-0.479865,-0.475856,0.004009,0.008884,0.436866,...,0.007399,0.007411,143,0.236667,0.002385,0.272563,0.708314,3,User_001,0
3,-0.476572,0.002982,-0.481614,-0.469691,-0.477044,-0.478612,-0.474071,0.004540,0.011923,0.259547,...,0.003517,0.004598,5,0.003333,0.259782,0.034869,0.351251,4,User_001,0
4,-0.476293,0.002300,-0.480955,-0.471099,-0.476830,-0.478196,-0.474352,0.003844,0.009855,0.258560,...,0.001567,0.001926,147,0.236667,0.004488,0.262486,0.903644,5,User_001,0


## 6. Cross-validation strategy

The competition metric is **macro F1-score**, so the validation metric must also be macro F1.

When possible, the notebook uses `GroupKFold` with the user ID as group.  
This is stricter than a simple random split because it tests whether the model generalizes to users not seen during training.

If there are not enough user groups, the notebook falls back to `StratifiedKFold`.


In [6]:
# ============================================================
# 6. Cross-validation helpers
# ============================================================

def split_features_labels(feature_df):
    """Split metadata and model features."""
    metadata_cols = ["file_id", "user_id", "label"]
    feature_cols = [c for c in feature_df.columns if c not in metadata_cols]

    X = feature_df[feature_cols].replace([np.inf, -np.inf], np.nan)
    y = feature_df["label"].astype(int)
    groups = feature_df["user_id"].astype(str)

    return X, y, groups, feature_cols


def get_cv_splits(X, y, groups, n_splits=5):
    """Use GroupKFold when possible, otherwise StratifiedKFold."""
    n_unique_groups = groups.nunique()
    min_class_count = y.value_counts().min()

    if n_unique_groups >= n_splits:
        print(f"Using GroupKFold with {n_splits} splits.")
        cv = GroupKFold(n_splits=n_splits)
        return list(cv.split(X, y, groups)), "GroupKFold"

    safe_splits = min(n_splits, min_class_count)
    if safe_splits < 2:
        raise ValueError("Not enough samples per class to run cross-validation.")

    print(f"Using StratifiedKFold with {safe_splits} splits.")
    cv = StratifiedKFold(n_splits=safe_splits, shuffle=True, random_state=RANDOM_STATE)
    return list(cv.split(X, y)), "StratifiedKFold"


def make_models():
    """Return a dictionary of candidate models."""
    models = {
        "RandomForest": RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=700,
            max_depth=None,
            min_samples_leaf=1,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        "HistGradientBoosting": HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.05,
            l2_regularization=0.01,
            random_state=RANDOM_STATE
        )
    }

    # Optional: use XGBoost if it is already installed.
    try:
        from xgboost import XGBClassifier
        models["XGBoost"] = XGBClassifier(
            n_estimators=500,
            max_depth=4,
            learning_rate=0.03,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        print("XGBoost detected and added to candidate models.")
    except Exception:
        print("XGBoost not installed. Skipping XGBoost model.")

    return models


def evaluate_feature_set(feature_df, feature_set_name):
    """Evaluate several models on one feature table using macro F1."""
    X, y, groups, feature_cols = split_features_labels(feature_df)
    splits, cv_name = get_cv_splits(X, y, groups)

    results = []
    oof_predictions = {}

    for model_name, model in make_models().items():
        fold_scores = []
        oof = np.zeros(len(y), dtype=int)

        for fold, (train_idx, valid_idx) in enumerate(splits, start=1):
            X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

            pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("model", model)
            ])

            pipeline.fit(X_train, y_train)
            pred = pipeline.predict(X_valid)

            score = f1_score(y_valid, pred, average="macro")
            fold_scores.append(score)
            oof[valid_idx] = pred

            print(f"{feature_set_name} | {model_name} | Fold {fold}: macro F1 = {score:.5f}")

        mean_score = np.mean(fold_scores)
        std_score = np.std(fold_scores)
        global_oof_score = f1_score(y, oof, average="macro")

        results.append({
            "feature_set": feature_set_name,
            "model": model_name,
            "cv": cv_name,
            "mean_macro_f1": mean_score,
            "std_macro_f1": std_score,
            "oof_macro_f1": global_oof_score
        })

        oof_predictions[(feature_set_name, model_name)] = oof

        print(f">>> {feature_set_name} | {model_name}: mean = {mean_score:.5f}, std = {std_score:.5f}, OOF = {global_oof_score:.5f}\n")

    return pd.DataFrame(results), oof_predictions


## 7. Baseline comparison

This section evaluates the three feature sets and compares them directly with the same cross-validation strategy.

The comparison acts as an ablation study:

- Baseline 1 measures the strength of simple global statistics.
- Baseline 2 shows whether temporal chunks, trends, autocorrelation and FFT improve the score.
- Baseline 3 tests whether stronger HAR-specific features such as magnitude, jerk and dominant frequencies bring additional performance.


In [7]:
# ============================================================
# 7. Evaluate Baseline 1, Baseline 2 and Baseline 3
# ============================================================

results_b1, oof_b1 = evaluate_feature_set(train_b1, "Baseline 1 - Statistical Features")
results_b2, oof_b2 = evaluate_feature_set(train_b2, "Baseline 2 - Temporal + FFT Features")
results_b3, oof_b3 = evaluate_feature_set(train_b3, "Baseline 3 - Improved HAR Features")

comparison_results = pd.concat([results_b1, results_b2, results_b3], ignore_index=True)
comparison_results = comparison_results.sort_values("mean_macro_f1", ascending=False).reset_index(drop=True)

display(comparison_results)

comparison_results.to_csv(OUTPUT_DIR / "baseline_comparison_results.csv", index=False)
print("Saved:", OUTPUT_DIR / "baseline_comparison_results.csv")


Using GroupKFold with 5 splits.
XGBoost detected and added to candidate models.
Baseline 1 - Statistical Features | RandomForest | Fold 1: macro F1 = 0.61482
Baseline 1 - Statistical Features | RandomForest | Fold 2: macro F1 = 0.69178
Baseline 1 - Statistical Features | RandomForest | Fold 3: macro F1 = 0.68025
Baseline 1 - Statistical Features | RandomForest | Fold 4: macro F1 = 0.63994
Baseline 1 - Statistical Features | RandomForest | Fold 5: macro F1 = 0.66134
>>> Baseline 1 - Statistical Features | RandomForest: mean = 0.65763, std = 0.02770, OOF = 0.67154

Baseline 1 - Statistical Features | ExtraTrees | Fold 1: macro F1 = 0.60518
Baseline 1 - Statistical Features | ExtraTrees | Fold 2: macro F1 = 0.67976
Baseline 1 - Statistical Features | ExtraTrees | Fold 3: macro F1 = 0.67096
Baseline 1 - Statistical Features | ExtraTrees | Fold 4: macro F1 = 0.62372
Baseline 1 - Statistical Features | ExtraTrees | Fold 5: macro F1 = 0.66338
>>> Baseline 1 - Statistical Features | ExtraTrees

,feature_set,model,cv,mean_macro_f1,std_macro_f1,oof_macro_f1
0,Baseline 3 - Improved HAR Features,XGBoost,GroupKFold,0.708339,0.028373,0.717338
1,Baseline 3 - Improved HAR Features,HistGradientBoosting,GroupKFold,0.703661,0.027473,0.710303
2,Baseline 2 - Temporal + FFT Features,XGBoost,GroupKFold,0.695652,0.025907,0.705927
3,Baseline 2 - Temporal + FFT Features,HistGradientBoosting,GroupKFold,0.692742,0.024997,0.700853
4,Baseline 1 - Statistical Features,XGBoost,GroupKFold,0.690010,0.029331,0.702220
5,Baseline 1 - Statistical Features,HistGradientBoosting,GroupKFold,0.683755,0.026542,0.694644
6,Baseline 1 - Statistical Features,RandomForest,GroupKFold,0.657625,0.027704,0.671543
7,Baseline 2 - Temporal + FFT Features,RandomForest,GroupKFold,0.653236,0.027939,0.663632
8,Baseline 2 - Temporal + FFT Features,ExtraTrees,GroupKFold,0.649158,0.027825,0.660034
9,Baseline 1 - Statistical Features,ExtraTrees,GroupKFold,0.648603,0.028961,0.663355


Saved: outputs\baseline_comparison_results.csv


## 8. Detailed analysis of the best validation model

We now select the best feature set and model according to cross-validation macro F1-score.

This is useful for the report because it provides:

- the selected model;
- the validation score;
- a classification report;
- a confusion matrix showing which classes are confused.


In [8]:
# ============================================================
# 8. Best model selection and OOF diagnostics
# ============================================================

best_row = comparison_results.iloc[0]
best_feature_set = best_row["feature_set"]
best_model_name = best_row["model"]

print("Best feature set:", best_feature_set)
print("Best model:", best_model_name)
print("Best mean macro F1:", best_row["mean_macro_f1"])

if best_feature_set.startswith("Baseline 1"):
    best_train_features = train_b1
    best_test_features = test_b1
    best_oof_dict = oof_b1
elif best_feature_set.startswith("Baseline 2"):
    best_train_features = train_b2
    best_test_features = test_b2
    best_oof_dict = oof_b2
else:
    best_train_features = train_b3
    best_test_features = test_b3
    best_oof_dict = oof_b3

X_best, y_best, groups_best, best_feature_cols = split_features_labels(best_train_features)
best_oof_pred = best_oof_dict[(best_feature_set, best_model_name)]

print("OOF classification report:")
print(classification_report(y_best, best_oof_pred, digits=4))

cm = confusion_matrix(y_best, best_oof_pred, labels=sorted(y_best.unique()))
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{c}" for c in sorted(y_best.unique())],
    columns=[f"pred_{c}" for c in sorted(y_best.unique())]
)
display(cm_df)

cm_df.to_csv(OUTPUT_DIR / "best_model_confusion_matrix.csv")
print("Saved:", OUTPUT_DIR / "best_model_confusion_matrix.csv")

Best feature set: Baseline 3 - Improved HAR Features
Best model: XGBoost
Best mean macro F1: 0.7083393136753331
OOF classification report:
              precision    recall  f1-score   support

           0     0.9623    0.9675    0.9649      4643
           1     0.8671    0.9393    0.9017      4695
           2     0.4444    0.0894    0.1488       358
           3     0.6808    0.7348    0.7067       656
           4     0.9453    0.8521    0.8963       142
           5     0.8464    0.5760    0.6855       526

    accuracy                         0.8929     11020
   macro avg     0.7910    0.6932    0.7173     11020
weighted avg     0.8824    0.8929    0.8819     11020



,pred_0,pred_1,pred_2,pred_3,pred_4,pred_5
true_0,4492,135,0,15,0,1
true_1,173,4410,18,71,0,23
true_2,2,251,32,66,0,7
true_3,1,131,12,482,7,23
true_4,0,1,0,19,121,1
true_5,0,158,10,55,0,303


Saved: outputs\best_model_confusion_matrix.csv


## 9. Feature importance

For tree-based models, feature importance can help explain which engineered features were useful.

This is especially helpful for the report section about preprocessing and method design.


In [9]:
# ============================================================
# 9. Train best model on full training data and inspect features
# ============================================================

def get_model_by_name(model_name):
    models = make_models()
    if model_name not in models:
        raise ValueError(f"Unknown model name: {model_name}")
    return models[model_name]


final_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", get_model_by_name(best_model_name))
])

final_model.fit(X_best, y_best)

model_step = final_model.named_steps["model"]

if hasattr(model_step, "feature_importances_"):
    importance_df = pd.DataFrame({
        "feature": best_feature_cols,
        "importance": model_step.feature_importances_
    }).sort_values("importance", ascending=False)

    display(importance_df.head(30))
    importance_df.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
    print("Saved:", OUTPUT_DIR / "feature_importance.csv")
else:
    print("This model does not expose feature_importances_.")


XGBoost detected and added to candidate models.


,feature,importance
51,std_x_q75,0.162300
81,std_z_q75,0.063616
66,std_y_q75,0.028881
408,jerk_mag_iqr,0.025103
60,std_y_mean,0.022387
65,std_y_q25,0.019969
658,std_y_energy_mean,0.014488
659,std_y_abs_mean,0.014238
660,std_y_rms,0.013056
273,std_y_fft_total_power,0.012970


Saved: outputs\feature_importance.csv


## 10. Generate Kaggle submission

The submission must strictly follow the format:

```text
Id,Label
11021,0
11022,3
...
```

Important details:

- `Id` must correspond to `file_id`.
- `Label` must be an integer from 0 to 5.
- The order should match `sample_submission.csv` when available.
- Any ID mismatch can cause an error or poor Kaggle performance.


In [10]:
# ============================================================
# 10. Kaggle submission generation
# ============================================================

X_test = best_test_features[best_feature_cols].replace([np.inf, -np.inf], np.nan)
test_pred = final_model.predict(X_test).astype(int)

pred_df = pd.DataFrame({
    "Id": best_test_features["file_id"].astype(int),
    "Label": test_pred
})

# Align with sample_submission.csv if available.
if SAMPLE_SUBMISSION_PATH.exists():
    sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    sample_submission.columns = [c.strip() for c in sample_submission.columns]

    if not {"Id", "Label"}.issubset(set(sample_submission.columns)):
        raise ValueError("sample_submission.csv must contain columns 'Id' and 'Label'.")

    sample_submission["Id"] = sample_submission["Id"].astype(int)
    pred_df["Id"] = pred_df["Id"].astype(int)

    submission = sample_submission[["Id"]].merge(pred_df, on="Id", how="left")

    missing_count = submission["Label"].isna().sum()
    if missing_count > 0:
        missing_ids = submission.loc[submission["Label"].isna(), "Id"].head(10).tolist()
        raise ValueError(f"Missing predictions for {missing_count} IDs. Example missing IDs: {missing_ids}")

    submission["Label"] = submission["Label"].astype(int)
else:
    print("Warning: sample_submission.csv not found. Submission will be sorted by Id.")
    submission = pred_df.sort_values("Id").reset_index(drop=True)

# Final sanity checks.
assert list(submission.columns) == ["Id", "Label"]
assert submission["Label"].between(0, 5).all(), "Predicted labels must be between 0 and 5."
assert submission["Id"].is_unique, "Submission IDs must be unique."

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

display(submission.head())
print("Submission shape:", submission.shape)
print("Saved submission to:", submission_path)


,Id,Label
0,11021,0
1,11022,0
2,11023,0
3,11024,0
4,11025,0


Submission shape: (6849, 2)
Saved submission to: outputs\submission.csv
